<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/03-true-streaming-causal-model/causal_types.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F

d_k is vector length (embedding dimension) of Q and K


d_k = Q.shape[-1] or K.shape[-1] by construction

In [ ]:
# usual bidirectional self-attention - sees everything
def attention(x, Wq, Wk, Wv):
  Q, K, V = x @ Wq, x @ Wk, x @ Wv
  d_k = Q.shape[-1]
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
  weights = F.softmax(scores, dim=-1)
  return weights @ V

In [ ]:
# causal attention - blocks the future
def causal_attention(x, Wq, Wk, Wv):
  Q, K, V = x @ Wq, x @ Wk, x @ Wv
  d_k = Q.shape[-1]
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)

  # x = (T, D)
  T = x.shape[0]
  future_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
  scores = scores.masked_fill(future_mask, float('-inf'))

  weights = F.softmax(scores, dim=-1)
  return weights @ V

In [ ]:
# kv cache causal attention
# x_new: (T_new, D) - only the newly arrived audio chunk
# k_cache, v_cache: (T_past, D), or None on the very first call
# returns output for just this chunk alongwith updated cache to pass into next call

# but this can cause memory to grow unboundedly that's why sliding window attention is preferred
def causal_attention_with_cache(x_new, Wq, Wk, Wv, k_cache=None, v_cache=None, max_cache=6):
  Q_new, K_new, V_new = x_new @ Wq, x_new @ Wk, x_new @ Wv

  if k_cache is None:
    K, V = K_new, V_new
  else:
    K = torch.cat([k_cache, K_new], dim=0)
    V = torch.cat([v_cache, V_new], dim=0)

  # if want to bound memory with most recent
  if K.shape[0] > max_cache:
    K = K[-max_cache:]
    V = V[-max_cache:]

  d_k = Q_new.shape[-1]
  scores = Q_new @ K.transpose(-1, -1) / (d_k ** 0.5)
  weights = F.softmax(scores, dim=-1)
  output = weights @ V
  return output, K, V

In [ ]:
# sliding window attention - blocks the future and the distant past
# window decides how mant steps back is 'distant'
def sliding_window_attention(x, Wq, Wk, Wv, window=3):
  Q, K, V = x @ Wq, x @ Wk, x @ Wv
  d_k = Q.shape[-1]
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)

  T = x.shape[0]
  future_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
  distant_past = torch.tril(torch.ones(T, T), diagonal=-window).bool()
  scores = scores.masked_fill(future_mask | distant_past, float('-inf'))

  weights = F.softmax(scores, dim=-1)
  return weights @ V